<a href="https://colab.research.google.com/github/Almas1989/PySpark_colab_practice/blob/main/Getting_Started_with_PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Начало работы с PySpark в Google Colab

PySpark - это Python интерфейс для Apache Spark. Основные сценарии использования PySpark - работа с огромными объемами данных и создание конвейеров обработки данных.

Подробнее здесь! http://spark.apache.org/docs/latest/api/python/

# 1. Установка PySpark в Google Colab

In [1]:
# ! в ipynb означает: выполнить команду в shell ОС, а не в Python.
# Обновляется список пакетов Linux.
# Устанавливается OpenJDK 8 (headless) — минимальная версия Java без GUI.
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

# Проверьте этот сайт для последней ссылки на загрузку https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
# Загрузка и распаковка Spark
!wget -q https://dlcdn.apache.org/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz

# Установка Python-библиотек
!pip install -q findspark 
!pip install pyspark
!pip install py4j


# Импорты и переменные окружения
import os
import sys
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
# os.environ["SPARK_HOME"] = "/content/spark-3.2.1-bin-hadoop3.2"

# findspark — связываем Python и Spark
import findspark
findspark.init()
findspark.find()


# Импорт Spark API
import pyspark
from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F


# Создание SparkSession — ключевой момент
spark= SparkSession \
       .builder \
       .appName("Наш первый Spark пример") \
       .getOrCreate()

spark

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,860 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,573 kB]  
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease   
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]     
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,205 kB]
Get:13 http://archive.ubuntu.com/ubun

In [2]:
spark

# 2. Чтение данных

Для этого примера я буду использовать публично доступный набор данных в формате CSV.

In [5]:
import requests
path = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv"
req = requests.get(path)
url_content = req.content

csv_file_name = 'owid-covid-data.csv'
csv_file = open(csv_file_name, 'wb')

csv_file.write(url_content)
csv_file.close()

# Чтение CSV через Spark
df = spark.read.csv('/content/'+csv_file_name, header=True, inferSchema=True)


# 3. PySpark DataFrames

In [6]:
# Просмотр схемы dataframe
df.printSchema()

root
 |-- iso_code: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- location: string (nullable = true)
 |-- date: date (nullable = true)
 |-- total_cases: integer (nullable = true)
 |-- new_cases: integer (nullable = true)
 |-- new_cases_smoothed: double (nullable = true)
 |-- total_deaths: integer (nullable = true)
 |-- new_deaths: integer (nullable = true)
 |-- new_deaths_smoothed: double (nullable = true)
 |-- total_cases_per_million: double (nullable = true)
 |-- new_cases_per_million: double (nullable = true)
 |-- new_cases_smoothed_per_million: double (nullable = true)
 |-- total_deaths_per_million: double (nullable = true)
 |-- new_deaths_per_million: double (nullable = true)
 |-- new_deaths_smoothed_per_million: double (nullable = true)
 |-- reproduction_rate: double (nullable = true)
 |-- icu_patients: integer (nullable = true)
 |-- icu_patients_per_million: double (nullable = true)
 |-- hosp_patients: integer (nullable = true)
 |-- hosp_patients_per_mil

In [8]:
# Преобразование колонки с датой
df.select(F.to_date("date", "yyyy-MM-dd").alias('date'))

DataFrame[date: date]

In [11]:
# Сводная статистика
df.describe().show()

+-------+--------+-------------+-----------+--------------------+------------------+------------------+------------------+------------------+-------------------+-----------------------+---------------------+------------------------------+------------------------+----------------------+-------------------------------+------------------+------------------+------------------------+------------------+-------------------------+---------------------+---------------------------------+----------------------+----------------------------------+-------------------+------------------+------------------------+----------------------+------------------+-------------------------------+-------------------+------------------+-------------+--------------------+--------------------+-----------------------+--------------------+------------------+-------------------------+------------------------------+-----------------------------+-----------------------------------+--------------------------+-------------

In [ ]:
# по одной колонке
df.select(
    F.count("total_cases").alias("cnt"),
    F.mean("total_cases").alias("avg"),
    F.min("total_cases").alias("min"),
    F.max("total_cases").alias("max")
).show()

+------+-----------------+---+---------+
|   cnt|              avg|min|      max|
+------+-----------------+---+---------+
|411804|7365292.354484173|  0|775866783|
+------+-----------------+---+---------+



In [ ]:
# по всем колонкам
df.summary("count", "mean", "min", "max").show()

+-------+--------+-------------+-----------+-----------------+-----------------+------------------+-----------------+----------------+-------------------+-----------------------+---------------------+------------------------------+------------------------+----------------------+-------------------------------+------------------+-----------------+------------------------+------------------+-------------------------+---------------------+---------------------------------+----------------------+----------------------------------+-------------------+-----------------+------------------------+----------------------+------------------+-------------------------------+-------------------+------------------+-------------+-------------------+--------------------+-----------------------+--------------------+-----------------+-------------------------+------------------------------+-----------------------------+-----------------------------------+--------------------------+------------------------

In [18]:
# Фильтрация DataFrame
df.filter(df.location == "United States").orderBy(F.desc("date")).show(10)

+--------+-------------+-------------+----------+-----------+---------+------------------+------------+----------+-------------------+-----------------------+---------------------+------------------------------+------------------------+----------------------+-------------------------------+-----------------+------------+------------------------+-------------+-------------------------+---------------------+---------------------------------+----------------------+----------------------------------+-----------+---------+------------------------+----------------------+------------------+-------------------------------+-------------+--------------+-----------+------------------+-----------------+-----------------------+--------------+----------------+-------------------------+------------------------------+-----------------------------+-----------------------------------+--------------------------+-------------------------------------+------------------------------+-------------------------

In [27]:
# Простая функция Group By
df.groupBy("location").sum("new_cases").orderBy(F.desc("sum(new_cases)")).show(10)

+--------------------+--------------+
|            location|sum(new_cases)|
+--------------------+--------------+
|               World|     775935057|
|High-income count...|     429044052|
|                Asia|     301564180|
|              Europe|     252916868|
|Upper-middle-inco...|     251756125|
| European Union (27)|     185822587|
|       North America|     124492698|
|       United States|     103436829|
|               China|      99373219|
|Lower-middle-inco...|      92019711|
+--------------------+--------------+
only showing top 10 rows


# 4. Spark SQL

Что мне действительно нравится в модуле SQL, так это то, что с его помощью очень легко взаимодействовать с данными, продолжая использовать Spark. Нужно меньше изучать, поскольку это в основном тот же синтаксис SQL, с которым вы уже могли быть знакомы.

In [28]:
# Создание таблицы из dataframe
df.createOrReplaceTempView("covid_data")  # временное представление
# df.saveAsTable("covid_data")  # Сохранить как таблицу
# df.write.mode("overwrite").saveAsTable("covid_data")  # Сохранить как таблицу и перезаписать, если существует

In [29]:

df2 = spark.sql("SELECT * from covid_data")
df2.printSchema()
df2.show(5)

root
 |-- iso_code: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- location: string (nullable = true)
 |-- date: date (nullable = true)
 |-- total_cases: integer (nullable = true)
 |-- new_cases: integer (nullable = true)
 |-- new_cases_smoothed: double (nullable = true)
 |-- total_deaths: integer (nullable = true)
 |-- new_deaths: integer (nullable = true)
 |-- new_deaths_smoothed: double (nullable = true)
 |-- total_cases_per_million: double (nullable = true)
 |-- new_cases_per_million: double (nullable = true)
 |-- new_cases_smoothed_per_million: double (nullable = true)
 |-- total_deaths_per_million: double (nullable = true)
 |-- new_deaths_per_million: double (nullable = true)
 |-- new_deaths_smoothed_per_million: double (nullable = true)
 |-- reproduction_rate: double (nullable = true)
 |-- icu_patients: integer (nullable = true)
 |-- icu_patients_per_million: double (nullable = true)
 |-- hosp_patients: integer (nullable = true)
 |-- hosp_patients_per_mil

In [ ]:
groupDF = spark.sql("SELECT location, count(*) from covid_data group by location")
# the same in spark
groupDF.show(5)

+--------+--------+
|location|count(1)|
+--------+--------+
|    Chad|    1674|
|Anguilla|    1674|
|Kiribati|    1674|
|  Guyana|    1674|
| Eritrea|    1674|
+--------+--------+
only showing top 5 rows


# 5. Пример с другим набором данных
Этот набор данных поставляется с вашей сессией Google Colab

In [34]:
df = spark.read.csv("/content/sample_data/california_housing_train.csv", header=True, inferSchema=True)

In [35]:
df.printSchema()

root
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- median_house_value: double (nullable = true)



In [36]:
# Вывести N строк
df.show(5)

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+
|  -114.31|   34.19|              15.0|     5612.0|        1283.0|    1015.0|     472.0|       1.4936|           66900.0|
|  -114.47|    34.4|              19.0|     7650.0|        1901.0|    1129.0|     463.0|         1.82|           80100.0|
|  -114.56|   33.69|              17.0|      720.0|         174.0|     333.0|     117.0|       1.6509|           85700.0|
|  -114.57|   33.64|              14.0|     1501.0|         337.0|     515.0|     226.0|       3.1917|           73400.0|
|  -114.57|   33.57|              20.0|     1454.0|         326.0|     624.0|     262.0|        1.925|           65500.0|
+---------+--------+----

In [37]:
df.count()

17000

In [38]:
df.select("housing_median_age","total_rooms").show(5)

+------------------+-----------+
|housing_median_age|total_rooms|
+------------------+-----------+
|              15.0|     5612.0|
|              19.0|     7650.0|
|              17.0|      720.0|
|              14.0|     1501.0|
|              20.0|     1454.0|
+------------------+-----------+
only showing top 5 rows


In [39]:
df.describe().show()

+-------+-------------------+------------------+------------------+-----------------+-----------------+------------------+-----------------+------------------+------------------+
|summary|          longitude|          latitude|housing_median_age|      total_rooms|   total_bedrooms|        population|       households|     median_income|median_house_value|
+-------+-------------------+------------------+------------------+-----------------+-----------------+------------------+-----------------+------------------+------------------+
|  count|              17000|             17000|             17000|            17000|            17000|             17000|            17000|             17000|             17000|
|   mean|-119.56210823529375|  35.6252247058827| 28.58935294117647|2643.664411764706|539.4108235294118|1429.5739411764705|501.2219411764706| 3.883578100000021|207300.91235294117|
| stddev| 2.0051664084260357|2.1373397946570867|12.586936981660406|2179.947071452777|421.4994515798648| 1

In [42]:
df.select('total_rooms').distinct().show(5)

+-----------+
|total_rooms|
+-----------+
|      934.0|
|     3980.0|
|     4142.0|
|      596.0|
|     1761.0|
+-----------+
only showing top 5 rows


In [50]:
from pyspark.sql import functions as F

In [51]:
# Вариант 2 (Рекомендуемый): Округление/Биннинг количества комнат для адекватной группировки
# Например, округлим комнаты до сотен, чтобы создать группы
# в SQL это выглядело бы так:
    # SELECT
    #     ROUND(total_rooms / 100) * 100 AS rooms_bin,
    #     AVG(housing_median_age) AS avg_age,
    #     COUNT(*) AS count
    # FROM
    #     df_table
    # GROUP BY
    #     ROUND(total_rooms / 100) * 100
    # ORDER BY
    #     rooms_bin;

test = df.withColumn('rooms_bin', F.round(F.col('total_rooms') / 100) * 100) \
         .groupBy('rooms_bin') \
         .agg(F.avg('housing_median_age').alias('avg_age'), F.count('*').alias('count')) \
         .orderBy('rooms_bin')

In [53]:
# Вывод первых 10 строк результата в pandas DataFrame
test.limit(5).toPandas()

,rooms_bin,avg_age,count
0,0.0,30.900000,40
1,100.0,31.794118,102
2,200.0,32.850467,107
3,300.0,30.387597,129
4,400.0,32.407407,162


In [ ]:
# Правильная версия: если пусто, верни 1. Count посчитает эти единицы.
# в SQL это выглядело бы так:
# SELECT
#     COUNT(CASE WHEN col1 IS NULL THEN 1 END) as col1,
#     COUNT(CASE WHEN col2 IS NULL THEN 1 END) as col2,
#     COUNT(CASE WHEN col3 IS NULL THEN 1 END) as col3
#     -- и так далее для ВСЕХ колонок таблицы автоматически
# FROM df_table;

df.select([F.count(F.when(F.isnull(c), 1)).alias(c) for c in df.columns]).show()

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+
|        0|       0|                 0|          0|             0|         0|         0|            0|                 0|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+



# Создание тестового Spark DataFrame

In [ ]:
data = [
    ('John', 'Smith', 1),
    ('Jane', 'Smith', 2),
    ('Jonas', 'Smith', 3),
]


columns = ["firstname", "lastname", "id"] 
df = spark.createDataFrame(data=data, schema=columns)
df.show()

+---------+--------+---+
|firstname|lastname| id|
+---------+--------+---+
|     John|   Smith|  1|
|     Jane|   Smith|  2|
|    Jonas|   Smith|  3|
+---------+--------+---+



# Советы и хитрости Spark

Это коллекция фрагментов кода для распространенных или сложных задач

## Pandas DataFrame в Spark DataFrame

In [59]:
import pandas as pd
import numpy as np

df = pd.DataFrame(np.random.randint(100,size=(1000, 3)),columns=['A','B','C'])
spark_df = spark.createDataFrame(df)
spark_df.show(5)

+---+---+---+
|  A|  B|  C|
+---+---+---+
| 59| 99| 58|
| 11| 32|  2|
| 57| 78| 75|
| 60| 39| 99|
| 22| 48| 40|
+---+---+---+
only showing top 5 rows


In [62]:
# Преобразование объектных колонок в pandas dataframe в строки
for i in df.select_dtypes(include='object').columns.tolist():
	df[i] = df[i].astype(str)

# Преобразование datetime в UTC
for i in [col for col in df.columns if df[col].dtype == 'datetime64[ns]']:
   df[i] = pd.to_datetime(df[i], utc=True)

# Замена nan и "None" в pandas dataframe на null в spark dataframe
spark_df = spark.createDataFrame(df).replace('None', None).replace(float('nan'), None)

## Оконные функции

In [63]:
data = [
        (1,'2021-01-01 10:00:00'),
        (1,'2021-01-01 11:00:00'),
        (1,'2021-01-01 12:00:00'),
        (2,'2021-01-01 12:00:00'),
        (2,'2021-01-01 13:00:00'),
        (2,'2021-01-01 14:00:00'),
]

columns = ["id","datetime"]
df = spark.createDataFrame(data=data, schema = columns)
df.createOrReplaceTempView("window_test")
df.show()

+---+-------------------+
| id|           datetime|
+---+-------------------+
|  1|2021-01-01 10:00:00|
|  1|2021-01-01 11:00:00|
|  1|2021-01-01 12:00:00|
|  2|2021-01-01 12:00:00|
|  2|2021-01-01 13:00:00|
|  2|2021-01-01 14:00:00|
+---+-------------------+



In [65]:
# Выбор минимума и максимума по определенной группе
spark.sql('''--sql
Select
  id,
  max(datetime) OVER (Partition BY id ORDER BY datetime) as max_date,
  min(datetime) OVER (Partition BY id ORDER BY datetime) as min_date,
  ROW_NUMBER() OVER (Partition BY id ORDER BY datetime) as row_number
FROM window_test
''').show()

+---+-------------------+-------------------+----------+
| id|           max_date|           min_date|row_number|
+---+-------------------+-------------------+----------+
|  1|2021-01-01 10:00:00|2021-01-01 10:00:00|         1|
|  1|2021-01-01 11:00:00|2021-01-01 10:00:00|         2|
|  1|2021-01-01 12:00:00|2021-01-01 10:00:00|         3|
|  2|2021-01-01 12:00:00|2021-01-01 12:00:00|         1|
|  2|2021-01-01 13:00:00|2021-01-01 12:00:00|         2|
|  2|2021-01-01 14:00:00|2021-01-01 12:00:00|         3|
+---+-------------------+-------------------+----------+



In [66]:
# Выбор номера строки или порядкового ранга для каждой строки в указанной группе.
# Это отлично подходит для подранжирования в таблице

spark.sql('''
Select
  id,
  datetime,
  ROW_NUMBER() OVER (Partition BY id ORDER BY datetime) as row_number
  FROM window_test
''').show()

+---+-------------------+----------+
| id|           datetime|row_number|
+---+-------------------+----------+
|  1|2021-01-01 10:00:00|         1|
|  1|2021-01-01 11:00:00|         2|
|  1|2021-01-01 12:00:00|         3|
|  2|2021-01-01 12:00:00|         1|
|  2|2021-01-01 13:00:00|         2|
|  2|2021-01-01 14:00:00|         3|
+---+-------------------+----------+



## Дедупликация данных путем возврата наиболее недавно обновленной строки с использованием оконной функции

In [67]:
data = [
        (1,'2021-01-01',100,'A'),
        (1,'2021-01-31',105,'A'),
        (2,'2021-02-04',160,'B'),
        (2,'2021-02-07',145,'B'),
]

columns = ["id","date","score","type"]
df = spark.createDataFrame(data=data, schema = columns)
df.createOrReplaceTempView("window_test")
df.show()

+---+----------+-----+----+
| id|      date|score|type|
+---+----------+-----+----+
|  1|2021-01-01|  100|   A|
|  1|2021-01-31|  105|   A|
|  2|2021-02-04|  160|   B|
|  2|2021-02-07|  145|   B|
+---+----------+-----+----+



In [68]:
df2 = spark.sql("""
WITH T AS (
  SELECT
  *,
  ROW_NUMBER() OVER (PARTITION BY id ORDER BY date DESC) AS version_number
  FROM window_test
)
SELECT * FROM T WHERE version_number = 1;
""")

df2.show()

+---+----------+-----+----+--------------+
| id|      date|score|type|version_number|
+---+----------+-----+----+--------------+
|  1|2021-01-31|  105|   A|             1|
|  2|2021-02-07|  145|   B|             1|
+---+----------+-----+----+--------------+



In [69]:
spark.sql("""
  SELECT
  *,
  SUM(score) OVER (PARTITION by type ORDER BY date) as score_cumulative
  FROM window_test

""").show()

+---+----------+-----+----+----------------+
| id|      date|score|type|score_cumulative|
+---+----------+-----+----+----------------+
|  1|2021-01-01|  100|   A|             100|
|  1|2021-01-31|  105|   A|             205|
|  2|2021-02-04|  160|   B|             160|
|  2|2021-02-07|  145|   B|             305|
+---+----------+-----+----+----------------+



## Ограничение количества результатов на группу с помощью оконной функции

In [70]:
import pandas as pd
import numpy as np

df = pd.DataFrame(
np.hstack((
    np.random.randint(1,5,size=(100000, 1)),
    np.random.randint(100,size=(100000, 1))
))
, columns=['company_id', 'number'])

dff = spark.createDataFrame(df)
dff.createOrReplaceTempView("window_test_limits")


In [72]:
spark.sql("""
WITH T AS (
  SELECT
    company_id,
    number,
    ROW_NUMBER() OVER (PARTITION BY company_id ORDER BY number) AS row_number
  FROM window_test_limits
    )

SELECT * FROM T WHERE row_number <= 100

""").show(10)

+----------+------+----------+
|company_id|number|row_number|
+----------+------+----------+
|         1|     0|         1|
|         1|     0|         2|
|         1|     0|         3|
|         1|     0|         4|
|         1|     0|         5|
|         1|     0|         6|
|         1|     0|         7|
|         1|     0|         8|
|         1|     0|         9|
|         1|     0|        10|
+----------+------+----------+
only showing top 10 rows


## Расчет 7-дневной скользящей средней

In [73]:
df = pd.DataFrame(pd.date_range('1/1/2022','1/31/2022',freq='D'), columns=['date'])
import random
df['company_id'] = 1
df['number'] = df.apply(lambda x: random.randint(0,100), axis = 1)

dff = spark.createDataFrame(df)
dff.createOrReplaceTempView("window_data")

dff.show()

+-------------------+----------+------+
|               date|company_id|number|
+-------------------+----------+------+
|2022-01-01 00:00:00|         1|    80|
|2022-01-02 00:00:00|         1|    10|
|2022-01-03 00:00:00|         1|    35|
|2022-01-04 00:00:00|         1|    69|
|2022-01-05 00:00:00|         1|    80|
|2022-01-06 00:00:00|         1|    87|
|2022-01-07 00:00:00|         1|    62|
|2022-01-08 00:00:00|         1|    85|
|2022-01-09 00:00:00|         1|    89|
|2022-01-10 00:00:00|         1|    44|
|2022-01-11 00:00:00|         1|    55|
|2022-01-12 00:00:00|         1|    51|
|2022-01-13 00:00:00|         1|    98|
|2022-01-14 00:00:00|         1|    58|
|2022-01-15 00:00:00|         1|    73|
|2022-01-16 00:00:00|         1|    53|
|2022-01-17 00:00:00|         1|    58|
|2022-01-18 00:00:00|         1|    94|
|2022-01-19 00:00:00|         1|    91|
|2022-01-20 00:00:00|         1|    83|
+-------------------+----------+------+
only showing top 20 rows


In [74]:
spark.sql("""
SELECT
  date,
  company_id,
  number,
  AVG(number) OVER (PARTITION BY company_id ORDER BY date ASC RANGE BETWEEN INTERVAL 6 DAYS PRECEDING AND CURRENT ROW) as last_7_day_avg
FROM window_data
""").show()

+-------------------+----------+------+------------------+
|               date|company_id|number|    last_7_day_avg|
+-------------------+----------+------+------------------+
|2022-01-01 00:00:00|         1|    80|              80.0|
|2022-01-02 00:00:00|         1|    10|              45.0|
|2022-01-03 00:00:00|         1|    35|41.666666666666664|
|2022-01-04 00:00:00|         1|    69|              48.5|
|2022-01-05 00:00:00|         1|    80|              54.8|
|2022-01-06 00:00:00|         1|    87|60.166666666666664|
|2022-01-07 00:00:00|         1|    62| 60.42857142857143|
|2022-01-08 00:00:00|         1|    85|61.142857142857146|
|2022-01-09 00:00:00|         1|    89| 72.42857142857143|
|2022-01-10 00:00:00|         1|    44| 73.71428571428571|
|2022-01-11 00:00:00|         1|    55| 71.71428571428571|
|2022-01-12 00:00:00|         1|    51| 67.57142857142857|
|2022-01-13 00:00:00|         1|    98| 69.14285714285714|
|2022-01-14 00:00:00|         1|    58| 68.5714285714285

## Месячные активные пользователи

In [75]:
import pandas as pd
df = pd.DataFrame(pd.date_range('1/1/2022','1/31/2022',freq='D'), columns=['login_date'])
import random
df['company_id'] = 1
df['user_id'] = df.apply(lambda x: random.randint(0,3), axis = 1)

dff = spark.createDataFrame(df)
dff.createOrReplaceTempView("users_data")

dff.show()

+-------------------+----------+-------+
|         login_date|company_id|user_id|
+-------------------+----------+-------+
|2022-01-01 00:00:00|         1|      3|
|2022-01-02 00:00:00|         1|      3|
|2022-01-03 00:00:00|         1|      2|
|2022-01-04 00:00:00|         1|      0|
|2022-01-05 00:00:00|         1|      1|
|2022-01-06 00:00:00|         1|      1|
|2022-01-07 00:00:00|         1|      0|
|2022-01-08 00:00:00|         1|      3|
|2022-01-09 00:00:00|         1|      3|
|2022-01-10 00:00:00|         1|      1|
|2022-01-11 00:00:00|         1|      1|
|2022-01-12 00:00:00|         1|      0|
|2022-01-13 00:00:00|         1|      1|
|2022-01-14 00:00:00|         1|      2|
|2022-01-15 00:00:00|         1|      1|
|2022-01-16 00:00:00|         1|      3|
|2022-01-17 00:00:00|         1|      3|
|2022-01-18 00:00:00|         1|      1|
|2022-01-19 00:00:00|         1|      2|
|2022-01-20 00:00:00|         1|      0|
+-------------------+----------+-------+
only showing top

In [76]:
# Пересмотреть эту трансформацию
spark.sql("""
SELECT
  login_date,
  COUNT(user_id) OVER (PARTITION BY login_date ORDER BY login_date ASC RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND CURRENT ROW) AS monthly_active_users
  FROM users_data
""").show()

+-------------------+--------------------+
|         login_date|monthly_active_users|
+-------------------+--------------------+
|2022-01-01 00:00:00|                   1|
|2022-01-02 00:00:00|                   1|
|2022-01-03 00:00:00|                   1|
|2022-01-04 00:00:00|                   1|
|2022-01-05 00:00:00|                   1|
|2022-01-06 00:00:00|                   1|
|2022-01-07 00:00:00|                   1|
|2022-01-08 00:00:00|                   1|
|2022-01-09 00:00:00|                   1|
|2022-01-10 00:00:00|                   1|
|2022-01-11 00:00:00|                   1|
|2022-01-12 00:00:00|                   1|
|2022-01-13 00:00:00|                   1|
|2022-01-14 00:00:00|                   1|
|2022-01-15 00:00:00|                   1|
|2022-01-16 00:00:00|                   1|
|2022-01-17 00:00:00|                   1|
|2022-01-18 00:00:00|                   1|
|2022-01-19 00:00:00|                   1|
|2022-01-20 00:00:00|                   1|
+----------

## Поиск разницы во времени между связанными строками с использованием оконной функции

In [77]:
data = [
        (1,'start','2021-01-01',100,'A'),
        (1,'end','2021-01-31',200,'A'),
        (2,'start','2021-03-05 4:53:11',100,'A'),
        (2,'end','2021-05-01 05:06:38',200,'A'),
]

columns = ["id","session","datetime","station_return","type"]
df = spark.createDataFrame(data=data, schema = columns)
df.createOrReplaceTempView("window_test")
df.show()

+---+-------+-------------------+--------------+----+
| id|session|           datetime|station_return|type|
+---+-------+-------------------+--------------+----+
|  1|  start|         2021-01-01|           100|   A|
|  1|    end|         2021-01-31|           200|   A|
|  2|  start| 2021-03-05 4:53:11|           100|   A|
|  2|    end|2021-05-01 05:06:38|           200|   A|
+---+-------+-------------------+--------------+----+



In [79]:
spark.sql('''
SELECT
  id,
  datetime,
  lead(datetime) OVER (PARTITION BY id ORDER BY datetime) as next_datetime,
  DATEDIFF(lead(datetime) OVER (PARTITION BY id ORDER BY datetime),datetime) as duration_in_days

FROM window_test

''').show()

+---+-------------------+-------------------+----------------+
| id|           datetime|      next_datetime|duration_in_days|
+---+-------------------+-------------------+----------------+
|  1|         2021-01-01|         2021-01-31|              30|
|  1|         2021-01-31|               NULL|            NULL|
|  2| 2021-03-05 4:53:11|2021-05-01 05:06:38|              57|
|  2|2021-05-01 05:06:38|               NULL|            NULL|
+---+-------------------+-------------------+----------------+



## Разворот (Unpivot)

In [78]:
from pyspark.sql.types import *


data = [
        ('tim', 10, 9, 8, 5),
        ('john', 5, 6, 3, 6),
        ('jane', 7, 8, 9, 10),

]

schema = StructType([
   StructField("name", StringType(), True),
   StructField("experience", IntegerType(), True),
   StructField("satisfaction", IntegerType(), True),
   StructField("customer_service", IntegerType(), True),
   StructField("speed_of_service", IntegerType(), True)])


df = spark.createDataFrame(data, schema=schema)

df.show()

+----+----------+------------+----------------+----------------+
|name|experience|satisfaction|customer_service|speed_of_service|
+----+----------+------------+----------------+----------------+
| tim|        10|           9|               8|               5|
|john|         5|           6|               3|               6|
|jane|         7|           8|               9|              10|
+----+----------+------------+----------------+----------------+



In [ ]:
# В SQL это выглядело бы так:
# SELECT name, question, score
# FROM df_table
# UNPIVOT (
#     score FOR question IN (
#         experience AS 'experience',
#         satisfaction AS 'satisfaction',
#         customer_service AS 'customer_service',
#         speed_of_service AS 'speed_of_service'
#     )
# );


cols = ['experience', 'satisfaction', 'customer_service', 'speed_of_service']

exprs = f"""stack({len(cols)}, {", ".join([f"'{i}',{i}" for i in cols])}) as (question,score)"""

unpivotted_df = df.select("name",F.expr(exprs))

unpivotted_df.show()

+----+----------------+-----+
|name|        question|score|
+----+----------------+-----+
| tim|      experience|   10|
| tim|    satisfaction|    9|
| tim|customer_service|    8|
| tim|speed_of_service|    5|
|john|      experience|    5|
|john|    satisfaction|    6|
|john|customer_service|    3|
|john|speed_of_service|    6|
|jane|      experience|    7|
|jane|    satisfaction|    8|
|jane|customer_service|    9|
|jane|speed_of_service|   10|
+----+----------------+-----+



## Замена значений с использованием словаря

In [81]:
df = (spark
    .createDataFrame([
        (1, 'hello',3),
        (2, 'hello',5),
        (3, 'hello',5),
        (135246, 'hello',4),
        (54936, 'hello',4)
        ],
        ["id", "text","num"]))

In [82]:
mapping = {
1: 5555,
4:9999
}

In [85]:

df.replace(mapping, 1, 'id').replace(mapping, 1, 'num').show()

# Или, если нужно применить замену ко всем колонкам сразу (без указания subset):
# df.replace(mapping, 1).show()

+------+-----+----+
|    id| text| num|
+------+-----+----+
|  5555|hello|   3|
|     2|hello|   5|
|     3|hello|   5|
|135246|hello|9999|
| 54936|hello|9999|
+------+-----+----+



## Создание диапазона дат

In [87]:
date_range_df = spark.sql("SELECT explode(sequence(to_date('2018-01-01'), to_date('2018-03-01'), interval 1 day)) as date")
date_range_df.show(10)

+----------+
|      date|
+----------+
|2018-01-01|
|2018-01-02|
|2018-01-03|
|2018-01-04|
|2018-01-05|
|2018-01-06|
|2018-01-07|
|2018-01-08|
|2018-01-09|
|2018-01-10|
+----------+
only showing top 10 rows


## Объединение значений строк после группировки

In [88]:
df = (spark
    .createDataFrame([
        (1, 'hello',3),
        (2, 'hello',5),
        (3, 'hello',5),
        (3, 'hello',5),
        (3, 'hello',5),
        ],
        ["id", "text"]))

df.createOrReplaceTempView("group_array")

df.show()

+---+-----+---+
| id| text| _3|
+---+-----+---+
|  1|hello|  3|
|  2|hello|  5|
|  3|hello|  5|
|  3|hello|  5|
|  3|hello|  5|
+---+-----+---+



In [89]:
# Вернуть каждый элемент
spark.sql("Select g.text, collect_list(g.id) FROM group_array as g GROUP BY 1").show()

+-----+----------------+
| text|collect_list(id)|
+-----+----------------+
|hello| [1, 2, 3, 3, 3]|
+-----+----------------+



In [90]:
# Вернуть уникальный список
spark.sql("Select g.text, collect_set(g.id) FROM group_array as g GROUP BY 1").show()

+-----+---------------+
| text|collect_set(id)|
+-----+---------------+
|hello|      [1, 2, 3]|
+-----+---------------+



## Переименование колонок Spark с помощью словаря

In [92]:
import os

# 1. Ссылка и локальный путь
url = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv"
local_filename = "owid-covid-data.csv"

# 2. Скачиваем файл (используем системный wget или curl, так как это быстрее python requests для больших файлов)
# В Jupyter/Colab используем !
if not os.path.exists(local_filename):
    os.system(f"wget -q {url} -O {local_filename}")

# 3. Читаем файл ПРАВИЛЬНЫМ методом (csv, а не parquet)
# header=True - чтобы первая строка стала заголовками
# inferSchema=True - чтобы Spark угадал типы (int, double, date)
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(local_filename)

# 4. Теперь применяем вашу логику выборки колонок
# Убедитесь, что список cols_2016 содержит реальные имена из этого CSV
# df = df.select(*cols_2016)  # Используйте * для распаковки списка, это чище чем списковое включение

# 5. Переименование (ваша логика)
col_dict = {
    'iso_code': 'ISO_CODE_RENAMED', # Пример реальной колонки из этого датасета
    'location': 'COUNTRY'
}

for old_name, new_name in col_dict.items():
    # Проверка, чтобы не падать, если колонки нет
    if old_name in df.columns:
        df = df.withColumnRenamed(old_name, new_name)

df.show(5)

+----------------+---------+-----------+----------+-----------+---------+------------------+------------+----------+-------------------+-----------------------+---------------------+------------------------------+------------------------+----------------------+-------------------------------+-----------------+------------+------------------------+-------------+-------------------------+---------------------+---------------------------------+----------------------+----------------------------------+-----------+---------+------------------------+----------------------+------------------+-------------------------------+-------------+--------------+-----------+------------------+-----------------+-----------------------+--------------+----------------+-------------------------+------------------------------+-----------------------------+-----------------------------------+--------------------------+-------------------------------------+------------------------------+-----------------------

## Чтение нескольких Parquet файлов в один Spark DataFrame

In [94]:
# 1. Сначала читаем CSV, который у нас УЖЕ есть
# (Предполагаем, что вы скачали его на предыдущем шаге)
df_csv = spark.read.option("header", "true").csv("/content/owid-covid-data.csv")

# 2. Сохраняем его как Parquet (Конвертация)
# mode("overwrite") перезапишет файл, если он уже есть
df_csv.write.mode("overwrite").parquet("/content/my_data.parquet")

# 3. А вот ТЕПЕРЬ ваш код с glob заработает
import glob
parquet_files = glob.glob('/content/*.parquet')

# Проверка: покажем, что файлы теперь найдены
print(f"Найденные файлы: {parquet_files}")

# Чтение
df = spark.read.parquet(*parquet_files)
df.show(5)

Найденные файлы: ['/content/my_data.parquet']
+--------+---------+-----------+----------+-----------+---------+------------------+------------+----------+-------------------+-----------------------+---------------------+------------------------------+------------------------+----------------------+-------------------------------+-----------------+------------+------------------------+-------------+-------------------------+---------------------+---------------------------------+----------------------+----------------------------------+-----------+---------+------------------------+----------------------+------------------+-------------------------------+-------------+--------------+-----------+------------------+-----------------+-----------------------+--------------+----------------+-------------------------+------------------------------+-----------------------------+-----------------------------------+--------------------------+-------------------------------------+----------------

## Разделение и получение последнего элемента в Spark SQL

In [95]:
spark.sql("""
SELECT
  "This.is.a.test" AS text,
  SPLIT("This.is.a.test",'[\.]') AS split,
  REVERSE(SPLIT("This.is.a.test",'[\.]'))[0] AS last_word
""").show()

+--------------+-------------------+---------+
|          text|              split|last_word|
+--------------+-------------------+---------+
|This.is.a.test|[This, is, a, test]|     test|
+--------------+-------------------+---------+



<>:4: SyntaxWarning: invalid escape sequence '\.'
<>:4: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipython-input-387237255.py:4: SyntaxWarning: invalid escape sequence '\.'
  SPLIT("This.is.a.test",'[\.]') AS split,


## Обработка NULL значений

In [96]:
df = (spark
    .createDataFrame([
        (1, 'hello',None),
        (2, 'hello',None),
        (3, 'hello',5),
        (3, 'hello',5),
        (3, 'hello',5),
        ],
        ["id", "text"]))

df.createOrReplaceTempView("group_array")

df.show()

+---+-----+----+
| id| text|  _3|
+---+-----+----+
|  1|hello|NULL|
|  2|hello|NULL|
|  3|hello|   5|
|  3|hello|   5|
|  3|hello|   5|
+---+-----+----+



In [97]:
spark.sql("Select * from group_array where _3 IS NOT NULL").show()

+---+-----+---+
| id| text| _3|
+---+-----+---+
|  3|hello|  5|
|  3|hello|  5|
|  3|hello|  5|
+---+-----+---+



In [100]:
# Если у вас уже есть spark session (переменная spark)
# Просто используйте её. Spark сам поднимет метастор (derby) и sql-движок.

spark.sql("CREATE TABLE IF NOT EXISTS test_table (id INT, name STRING)")
spark.sql("INSERT INTO test_table VALUES (1, 'Test Hive')")
spark.sql("SELECT * FROM test_table").show()

# Это работает ВНУТРИ Colab без всяких JDBC и внешних портов.

+---+---------+
| id|     name|
+---+---------+
|  1|Test Hive|
+---+---------+



# Регулярные выражения (Regex)

In [101]:
spark.sql("""
SELECT
  '(5) Strongly Agree',
  regexp_extract('(10) Strongly Agree', '([0-9]+)')
""").show()

+------------------+------------------------------------------------+
|(5) Strongly Agree|regexp_extract((10) Strongly Agree, ([0-9]+), 1)|
+------------------+------------------------------------------------+
|(5) Strongly Agree|                                              10|
+------------------+------------------------------------------------+



# User Defined Functions (UDF)

UDF позволяет создавать пользовательские функции для применения к данным в DataFrame

In [102]:
# Пример 1: Простая UDF для преобразования текста в верхний регистр
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType, IntegerType

# Определяем функцию Python
def to_upper(text):
    return text.upper() if text else None

# Регистрируем как UDF
upper_udf = udf(to_upper, StringType())

# Создаем тестовый DataFrame
test_data = [
    (1, 'hello'),
    (2, 'world'),
    (3, 'pyspark')
]
df_test = spark.createDataFrame(test_data, ["id", "text"])

# Применяем UDF
df_test.withColumn("text_upper", upper_udf(df_test.text)).show()

+---+-------+----------+
| id|   text|text_upper|
+---+-------+----------+
|  1|  hello|     HELLO|
|  2|  world|     WORLD|
|  3|pyspark|   PYSPARK|
+---+-------+----------+



In [103]:
# Пример 2: UDF с декоратором и регистрация для SQL
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

@udf(returnType=IntegerType())
def calculate_age_category(age):
    """Категоризация возраста"""
    if age is None:
        return None
    elif age < 18:
        return 1  # Молодой
    elif age < 60:
        return 2  # Взрослый
    else:
        return 3  # Пожилой

# Регистрируем UDF для использования в SQL
spark.udf.register("age_category_sql", calculate_age_category)

# Тестовые данные
age_data = [(1, 15), (2, 35), (3, 65), (4, 28)]
df_ages = spark.createDataFrame(age_data, ["id", "age"])

# Использование через DataFrame API
df_ages.withColumn("category", calculate_age_category(df_ages.age)).show()

# Использование через SQL
df_ages.createOrReplaceTempView("ages_table")
spark.sql("SELECT id, age, age_category_sql(age) as category FROM ages_table").show()

+---+---+--------+
| id|age|category|
+---+---+--------+
|  1| 15|       1|
|  2| 35|       2|
|  3| 65|       3|
|  4| 28|       2|
+---+---+--------+

+---+---+--------+
| id|age|category|
+---+---+--------+
|  1| 15|       1|
|  2| 35|       2|
|  3| 65|       3|
|  4| 28|       2|
+---+---+--------+



# Joins - Объединение DataFrame

Joins позволяют объединять несколько DataFrame по общим ключам. PySpark поддерживает различные типы объединений

In [104]:
# Создаем тестовые DataFrame для демонстрации Joins
# DataFrame с пользователями
users_data = [
    (1, 'Alice', 'IT'),
    (2, 'Bob', 'HR'),
    (3, 'Charlie', 'IT'),
    (4, 'Diana', 'Finance')
]
df_users = spark.createDataFrame(users_data, ["user_id", "name", "department"])

# DataFrame с зарплатами
salaries_data = [
    (1, 70000),
    (2, 60000),
    (3, 75000),
    (5, 80000)  # user_id 5 не существует в users
]
df_salaries = spark.createDataFrame(salaries_data, ["user_id", "salary"])

print("Users DataFrame:")
df_users.show()

print("Salaries DataFrame:")
df_salaries.show()

Users DataFrame:
+-------+-------+----------+
|user_id|   name|department|
+-------+-------+----------+
|      1|  Alice|        IT|
|      2|    Bob|        HR|
|      3|Charlie|        IT|
|      4|  Diana|   Finance|
+-------+-------+----------+

Salaries DataFrame:
+-------+------+
|user_id|salary|
+-------+------+
|      1| 70000|
|      2| 60000|
|      3| 75000|
|      5| 80000|
+-------+------+



In [105]:
# 1. INNER JOIN - возвращает только совпадающие записи из обоих DataFrame
print("INNER JOIN:")
df_users.join(df_salaries, on="user_id", how="inner").show()

# 2. LEFT JOIN (LEFT OUTER) - все записи из левого DF + совпадения из правого
print("\nLEFT JOIN:")
df_users.join(df_salaries, on="user_id", how="left").show()

# 3. RIGHT JOIN (RIGHT OUTER) - все записи из правого DF + совпадения из левого
print("\nRIGHT JOIN:")
df_users.join(df_salaries, on="user_id", how="right").show()

# 4. FULL OUTER JOIN - все записи из обоих DataFrame
print("\nFULL OUTER JOIN:")
df_users.join(df_salaries, on="user_id", how="outer").show()

INNER JOIN:
+-------+-------+----------+------+
|user_id|   name|department|salary|
+-------+-------+----------+------+
|      1|  Alice|        IT| 70000|
|      2|    Bob|        HR| 60000|
|      3|Charlie|        IT| 75000|
+-------+-------+----------+------+


LEFT JOIN:
+-------+-------+----------+------+
|user_id|   name|department|salary|
+-------+-------+----------+------+
|      1|  Alice|        IT| 70000|
|      2|    Bob|        HR| 60000|
|      3|Charlie|        IT| 75000|
|      4|  Diana|   Finance|  NULL|
+-------+-------+----------+------+


RIGHT JOIN:
+-------+-------+----------+------+
|user_id|   name|department|salary|
+-------+-------+----------+------+
|      1|  Alice|        IT| 70000|
|      2|    Bob|        HR| 60000|
|      5|   NULL|      NULL| 80000|
|      3|Charlie|        IT| 75000|
+-------+-------+----------+------+


FULL OUTER JOIN:
+-------+-------+----------+------+
|user_id|   name|department|salary|
+-------+-------+----------+------+
|     

In [106]:
# Joins с использованием SQL
df_users.createOrReplaceTempView("users")
df_salaries.createOrReplaceTempView("salaries")

print("INNER JOIN через SQL:")
spark.sql("""
    SELECT u.user_id, u.name, u.department, s.salary
    FROM users u
    INNER JOIN salaries s ON u.user_id = s.user_id
""").show()

print("\nLEFT JOIN через SQL:")
spark.sql("""
    SELECT u.user_id, u.name, u.department, s.salary
    FROM users u
    LEFT JOIN salaries s ON u.user_id = s.user_id
""").show()

INNER JOIN через SQL:
+-------+-------+----------+------+
|user_id|   name|department|salary|
+-------+-------+----------+------+
|      1|  Alice|        IT| 70000|
|      2|    Bob|        HR| 60000|
|      3|Charlie|        IT| 75000|
+-------+-------+----------+------+


LEFT JOIN через SQL:
+-------+-------+----------+------+
|user_id|   name|department|salary|
+-------+-------+----------+------+
|      1|  Alice|        IT| 70000|
|      2|    Bob|        HR| 60000|
|      3|Charlie|        IT| 75000|
|      4|  Diana|   Finance|  NULL|
+-------+-------+----------+------+



# Работа с JSON данными

PySpark предоставляет мощные инструменты для работы с JSON: чтение файлов, парсинг JSON-строк и распаковка вложенных структур

In [107]:
# Пример 1: Создание DataFrame с JSON-строками
from pyspark.sql.functions import from_json, to_json, col, explode
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType

# Тестовые данные с JSON-строками
json_data = [
    (1, '{"name": "John", "age": 30, "city": "New York"}'),
    (2, '{"name": "Jane", "age": 25, "city": "Paris"}'),
    (3, '{"name": "Bob", "age": 35, "city": "London"}')
]
df_json_strings = spark.createDataFrame(json_data, ["id", "json_str"])

print("DataFrame с JSON-строками:")
df_json_strings.show(truncate=False)

DataFrame с JSON-строками:
+---+-----------------------------------------------+
|id |json_str                                       |
+---+-----------------------------------------------+
|1  |{"name": "John", "age": 30, "city": "New York"}|
|2  |{"name": "Jane", "age": 25, "city": "Paris"}   |
|3  |{"name": "Bob", "age": 35, "city": "London"}   |
+---+-----------------------------------------------+



In [108]:
# Пример 2: Парсинг JSON-строк в структурированные колонки
# Определяем схему JSON
json_schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("city", StringType(), True)
])

# Распаковываем JSON в отдельные колонки
df_parsed = df_json_strings.withColumn("parsed_data", from_json(col("json_str"), json_schema))

print("Распакованный JSON:")
df_parsed.select("id", "parsed_data.*").show()

# Преобразование обратно в JSON
df_to_json = df_parsed.select("id", to_json(col("parsed_data")).alias("json_output"))
print("\nОбратное преобразование в JSON:")
df_to_json.show(truncate=False)

Распакованный JSON:
+---+----+---+--------+
| id|name|age|    city|
+---+----+---+--------+
|  1|John| 30|New York|
|  2|Jane| 25|   Paris|
|  3| Bob| 35|  London|
+---+----+---+--------+


Обратное преобразование в JSON:
+---+------------------------------------------+
|id |json_output                               |
+---+------------------------------------------+
|1  |{"name":"John","age":30,"city":"New York"}|
|2  |{"name":"Jane","age":25,"city":"Paris"}   |
|3  |{"name":"Bob","age":35,"city":"London"}   |
+---+------------------------------------------+



In [109]:
# Пример 3: Работа с вложенными JSON и массивами
nested_json_data = [
    (1, '{"name": "Alice", "skills": ["Python", "Spark", "SQL"]}'),
    (2, '{"name": "Bob", "skills": ["Java", "Kafka"]}'),
    (3, '{"name": "Charlie", "skills": ["Python", "Machine Learning", "R"]}')
]
df_nested = spark.createDataFrame(nested_json_data, ["id", "json_str"])

# Схема с массивом
nested_schema = StructType([
    StructField("name", StringType(), True),
    StructField("skills", ArrayType(StringType()), True)
])

# Парсинг и распаковка массива
df_nested_parsed = df_nested.withColumn("data", from_json(col("json_str"), nested_schema))

print("Вложенный JSON с массивом:")
df_nested_parsed.select("id", "data.*").show(truncate=False)

# Используем explode для развертывания массива
print("\nРазвернутый массив навыков:")
df_nested_parsed.select("id", "data.name", explode("data.skills").alias("skill")).show()

Вложенный JSON с массивом:
+---+-------+-----------------------------+
|id |name   |skills                       |
+---+-------+-----------------------------+
|1  |Alice  |[Python, Spark, SQL]         |
|2  |Bob    |[Java, Kafka]                |
|3  |Charlie|[Python, Machine Learning, R]|
+---+-------+-----------------------------+


Развернутый массив навыков:
+---+-------+----------------+
| id|   name|           skill|
+---+-------+----------------+
|  1|  Alice|          Python|
|  1|  Alice|           Spark|
|  1|  Alice|             SQL|
|  2|    Bob|            Java|
|  2|    Bob|           Kafka|
|  3|Charlie|          Python|
|  3|Charlie|Machine Learning|
|  3|Charlie|               R|
+---+-------+----------------+



# Caching и Persistence

Кэширование позволяет сохранять промежуточные результаты в памяти или на диске для повторного использования, что значительно ускоряет вычисления

In [110]:
# Создаем большой DataFrame для демонстрации кэширования
import time

large_data = [(i, f"name_{i}", i * 100) for i in range(1, 10001)]
df_large = spark.createDataFrame(large_data, ["id", "name", "value"])

print("DataFrame создан")

# Без кэширования - каждый раз происходит пересчет
start_time = time.time()
count1 = df_large.filter(col("value") > 50000).count()
time1 = time.time() - start_time

start_time = time.time()
count2 = df_large.filter(col("value") > 50000).count()
time2 = time.time() - start_time

print(f"\nБез кэша:")
print(f"Первый запрос: {time1:.4f} сек, результат: {count1}")
print(f"Второй запрос: {time2:.4f} сек, результат: {count2}")

DataFrame создан

Без кэша:
Первый запрос: 0.4157 сек, результат: 9500
Второй запрос: 0.4588 сек, результат: 9500


In [111]:
# Теперь с кэшированием
df_cached = df_large.cache()  # или df_large.persist()

# Первый запрос - данные кэшируются
start_time = time.time()
count3 = df_cached.filter(col("value") > 50000).count()
time3 = time.time() - start_time

# Второй запрос - данные берутся из кэша (быстрее!)
start_time = time.time()
count4 = df_cached.filter(col("value") > 50000).count()
time4 = time.time() - start_time

print(f"\nС кэшем:")
print(f"Первый запрос (кэширование): {time3:.4f} сек, результат: {count3}")
print(f"Второй запрос (из кэша): {time4:.4f} сек, результат: {count4}")
print(f"Ускорение: {time3/time4:.2f}x")

# Очистка кэша
df_cached.unpersist()
print("\nКэш очищен")


С кэшем:
Первый запрос (кэширование): 0.9179 сек, результат: 9500
Второй запрос (из кэша): 0.1302 сек, результат: 9500
Ускорение: 7.05x

Кэш очищен


In [112]:
# Различные уровни persistence
from pyspark import StorageLevel

# MEMORY_ONLY - только в памяти (по умолчанию для cache())
df_memory_only = df_large.persist(StorageLevel.MEMORY_ONLY)

# MEMORY_AND_DISK - сначала память, потом диск
df_memory_disk = df_large.persist(StorageLevel.MEMORY_AND_DISK)

# DISK_ONLY - только на диске
df_disk_only = df_large.persist(StorageLevel.DISK_ONLY)

print("Доступные уровни persistence:")
print("- MEMORY_ONLY: данные только в памяти")
print("- MEMORY_AND_DISK: данные в памяти и на диске")  
print("- DISK_ONLY: данные только на диске")
print("- MEMORY_ONLY_SER: сериализованные данные в памяти")
print("- MEMORY_AND_DISK_SER: сериализованные данные в памяти и на диске")

# Очистка
df_memory_only.unpersist()
df_memory_disk.unpersist()
df_disk_only.unpersist()

Доступные уровни persistence:
- MEMORY_ONLY: данные только в памяти
- MEMORY_AND_DISK: данные в памяти и на диске
- DISK_ONLY: данные только на диске
- MEMORY_ONLY_SER: сериализованные данные в памяти
- MEMORY_AND_DISK_SER: сериализованные данные в памяти и на диске


DataFrame[id: bigint, name: string, value: bigint]

# Partitioning - Разделение данных

Partitioning позволяет распределить данные по нескольким разделам для параллельной обработки и оптимизации производительности

In [113]:
# Создаем DataFrame для демонстрации partitioning
partition_data = [(i, f"category_{i % 5}", i * 10) for i in range(1, 101)]
df_part = spark.createDataFrame(partition_data, ["id", "category", "value"])

# Проверяем текущее количество разделов
print(f"Текущее количество разделов: {df_part.rdd.getNumPartitions()}")

# Repartition - увеличиваем количество разделов (полная перетасовка данных)
df_repartitioned = df_part.repartition(10)
print(f"После repartition(10): {df_repartitioned.rdd.getNumPartitions()} разделов")

# Coalesce - уменьшаем количество разделов (без полной перетасовки, эффективнее)
df_coalesced = df_repartitioned.coalesce(5)
print(f"После coalesce(5): {df_coalesced.rdd.getNumPartitions()} разделов")

Текущее количество разделов: 2
После repartition(10): 10 разделов
После coalesce(5): 5 разделов


In [114]:
# Partitioning по колонке - данные с одинаковым значением попадут в один раздел
df_partitioned_by_col = df_part.repartition(5, "category")
print(f"\nРазделение по колонке 'category': {df_partitioned_by_col.rdd.getNumPartitions()} разделов")

# Это полезно для оптимизации группировок и joins
# Данные одной категории будут в одном разделе
print("\nПример данных после разделения по category:")
df_partitioned_by_col.show(10)

# Можно посмотреть распределение данных по разделам
def count_partition(iterator):
    yield sum(1 for _ in iterator)

partition_counts = df_partitioned_by_col.rdd.mapPartitions(count_partition).collect()
print(f"\nРаспределение записей по разделам: {partition_counts}")


Разделение по колонке 'category': 5 разделов

Пример данных после разделения по category:
+---+----------+-----+
| id|  category|value|
+---+----------+-----+
|  1|category_1|   10|
|  4|category_4|   40|
|  5|category_0|   50|
|  6|category_1|   60|
|  9|category_4|   90|
| 10|category_0|  100|
| 11|category_1|  110|
| 14|category_4|  140|
| 15|category_0|  150|
| 16|category_1|  160|
+---+----------+-----+
only showing top 10 rows

Распределение записей по разделам: [0, 0, 60, 20, 20]


# Broadcast Variables

Broadcast переменные позволяют эффективно распространять небольшие данные (справочники, lookup таблицы) на все узлы кластера для оптимизации joins

In [115]:
# Пример: большая таблица транзакций и маленький справочник категорий
from pyspark.sql.functions import broadcast

# Большая таблица транзакций
transactions_data = [(i, i % 10, i * 100) for i in range(1, 10001)]
df_transactions = spark.createDataFrame(transactions_data, ["transaction_id", "category_id", "amount"])

# Маленькая таблица-справочник категорий
categories_data = [
    (1, 'Electronics'),
    (2, 'Clothing'),
    (3, 'Food'),
    (4, 'Books'),
    (5, 'Toys'),
    (6, 'Sports'),
    (7, 'Home'),
    (8, 'Beauty'),
    (9, 'Automotive')
]
df_categories = spark.createDataFrame(categories_data, ["category_id", "category_name"])

print("Большая таблица транзакций:")
df_transactions.show(5)
print(f"Количество записей: {df_transactions.count()}")

print("\nМаленькая таблица категорий:")
df_categories.show()
print(f"Количество записей: {df_categories.count()}")

Большая таблица транзакций:
+--------------+-----------+------+
|transaction_id|category_id|amount|
+--------------+-----------+------+
|             1|          1|   100|
|             2|          2|   200|
|             3|          3|   300|
|             4|          4|   400|
|             5|          5|   500|
+--------------+-----------+------+
only showing top 5 rows
Количество записей: 10000

Маленькая таблица категорий:
+-----------+-------------+
|category_id|category_name|
+-----------+-------------+
|          1|  Electronics|
|          2|     Clothing|
|          3|         Food|
|          4|        Books|
|          5|         Toys|
|          6|       Sports|
|          7|         Home|
|          8|       Beauty|
|          9|   Automotive|
+-----------+-------------+

Количество записей: 9


In [116]:
# Обычный join (без broadcast)
print("Обычный JOIN:")
df_regular_join = df_transactions.join(df_categories, on="category_id", how="inner")
df_regular_join.show(10)

# Broadcast join - оптимизированный для маленьких таблиц
print("\nBROADCAST JOIN (оптимизированный):")
df_broadcast_join = df_transactions.join(broadcast(df_categories), on="category_id", how="inner")
df_broadcast_join.show(10)

# Broadcast join работает быстрее, потому что:
# 1. Маленькая таблица копируется на все узлы кластера один раз
# 2. Не требуется shuffle больших данных
# 3. Join происходит локально на каждом узле

print("\n✓ Broadcast join эффективен когда:")
print("  - Одна таблица намного меньше другой (обычно < 10MB)")
print("  - Нужно избежать дорогостоящей shuffle операции")
print("  - Join с lookup/справочными таблицами")

Обычный JOIN:
+-----------+--------------+------+-------------+
|category_id|transaction_id|amount|category_name|
+-----------+--------------+------+-------------+
|          1|             1|   100|  Electronics|
|          1|            11|  1100|  Electronics|
|          1|            21|  2100|  Electronics|
|          1|            31|  3100|  Electronics|
|          1|            41|  4100|  Electronics|
|          1|            51|  5100|  Electronics|
|          1|            61|  6100|  Electronics|
|          1|            71|  7100|  Electronics|
|          1|            81|  8100|  Electronics|
|          1|            91|  9100|  Electronics|
+-----------+--------------+------+-------------+
only showing top 10 rows

BROADCAST JOIN (оптимизированный):
+-----------+--------------+------+-------------+
|category_id|transaction_id|amount|category_name|
+-----------+--------------+------+-------------+
|          1|             1|   100|  Electronics|
|          2|            

In [117]:
# Пример использования broadcast переменной напрямую (не для joins)
# Полезно для передачи конфигурации, констант или небольших справочников

# Создаем broadcast переменную
category_mapping = {1: 'Electronics', 2: 'Clothing', 3: 'Food', 4: 'Books', 5: 'Toys'}
broadcast_mapping = spark.sparkContext.broadcast(category_mapping)

# Используем в UDF
from pyspark.sql.functions import udf

@udf(returnType=StringType())
def get_category_name(category_id):
    # Доступ к broadcast переменной через .value
    return broadcast_mapping.value.get(category_id, 'Unknown')

# Применяем
df_with_category = df_transactions.withColumn("category_name", get_category_name(col("category_id")))
print("Использование broadcast переменной в UDF:")
df_with_category.show(10)

# Освобождаем ресурсы
broadcast_mapping.unpersist()

Использование broadcast переменной в UDF:
+--------------+-----------+------+-------------+
|transaction_id|category_id|amount|category_name|
+--------------+-----------+------+-------------+
|             1|          1|   100|  Electronics|
|             2|          2|   200|     Clothing|
|             3|          3|   300|         Food|
|             4|          4|   400|        Books|
|             5|          5|   500|         Toys|
|             6|          6|   600|      Unknown|
|             7|          7|   700|      Unknown|
|             8|          8|   800|      Unknown|
|             9|          9|   900|      Unknown|
|            10|          0|  1000|      Unknown|
+--------------+-----------+------+-------------+
only showing top 10 rows
